# Experimento 05: TabPFN, um modelo de fundação para tabelas, sozinho e dentro do stacking

**O que é o TabPFN.** É uma rede *transformer* pré-treinada pela Prior Labs em milhões de tabelas sintéticas. Ela não "treina" no sentido usual: recebe os e-mails de treino como contexto e prevê os novos numa única passada (*in-context learning*). Versões usadas:
- **TabPFN-3.5-Plus** na nuvem (`tabpfn-client`), nas 20 divisões e na base externa;
- **TabPFN-3.5-Fast** local, em CPU, nas 5 divisões da semente 201, como réplica.

**Perguntas**
1. Sozinho, o TabPFN chega perto do Stacking 7?
2. Como 8ª família, ele melhora o stacking? Ou seja, erra em e-mails diferentes dos outros 7?
3. O resultado se mantém nas outras caixas de e-mail (Enron2–6)?

**Protocolo.** As mesmas 20 divisões dos experimentos 03 e 04 (sementes 201 a 204), os mesmos treinos internos e o mesmo limiar de custo (1 e-mail real perdido = 9 spams). As notas dos 7 modelos vêm do cache do exp. 03. As do TabPFN foram geradas por `scripts/tabpfn_notas.py`, com as divisões exportadas por índice.

**Termos.** *E-mail real* = e-mail legítimo. `real_lixeira` = % de e-mails reais marcados como spam. `spam_passou` = % de spam que chegou à caixa de entrada.

In [1]:
import os, gc, time, hashlib, platform, numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
from scipy import sparse, stats
from joblib import Parallel, delayed, parallel_config
import psutil, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import matthews_corrcoef, average_precision_score
from lightgbm import LGBMClassifier
import baycomp, sklearn

SMOKE = bool(os.environ.get("GA_SMOKE"))          # execucao reduzida so para testar o codigo
SEED, N_THREADS = 42, 4
CUSTO_FP, CUSTO_FN = 9, 1                          # 1 e-mail real perdido = 9 spams recebidos
CACHE = Path("cache_ga_smoke" if SMOKE else "../03_ga/cache_ga").resolve(); CACHE.mkdir(exist_ok=True)
LOG = Path("progresso_ga_smoke.log" if SMOKE else "progresso_analise.log")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

def log(msg):
    print(msg, flush=True)
    with LOG.open("a", encoding="utf-8") as fh: fh.write(time.strftime("%H:%M:%S ") + msg + "\n")
log(f"inicio | python {platform.python_version()} | sklearn {sklearn.__version__} | SMOKE={SMOKE}")

inicio | python 3.11.9 | sklearn 1.9.1 | SMOKE=False


## 1. Dados e a mesma base de teste trancada do experimento 02

In [2]:
RAIZ = next(p for p in [Path("../.."), Path(".."), Path(".")] if (p / "emails (1).csv").exists() or (p / "emails.csv").exists())
CSV = next(p for p in [RAIZ / "emails.csv", RAIZ / "emails (1).csv"] if p.exists())
df = pd.read_csv(CSV); y_all = df.pop("Prediction").values; df.pop("Email No.")
g_all = pd.util.hash_pandas_object(df, index=False).values
A = np.ascontiguousarray(df.values, dtype=np.int64)
hash_csv = {hashlib.md5(r.tobytes()).digest() for r in A}
X_all = sparse.csr_matrix(A.astype(np.float32)); del df, A; gc.collect()

ix = np.load(next(p for p in [Path("../indices_teste.npz"), Path("indices_teste.npz")] if p.exists()))
dev, teste = ix["dev"], ix["teste"]
assert len(dev) + len(teste) == len(y_all) and not set(g_all[dev]) & set(g_all[teste])
X, y, g = X_all[dev], y_all[dev], g_all[dev]
BASES = {"dev": (X, y, g), "all": (X_all, y_all, g_all)}
log(f"desenvolvimento {len(dev)} | teste trancado {len(teste)} ({(y_all[teste] == 0).sum()} reais, {(y_all[teste] == 1).sum()} spam)")

desenvolvimento 4137 | teste trancado 1035 (735 reais, 300 spam)


## 2. Espaço de busca: a árvore de parâmetros prováveis

In [3]:
tf = lambda: TfidfTransformer(sublinear_tf=True)
def lr(C, cw):          return make_pipeline(tf(), LogisticRegression(C=C, class_weight=cw, max_iter=5000))
def svm(k, C):          return make_pipeline(tf(), TruncatedSVD(k, random_state=SEED), Normalizer(), SVC(C=C, class_weight="balanced"))
def knn(k, w):          return make_pipeline(tf(), KNeighborsClassifier(k, weights=w, metric="cosine"))
def mlp(h, a):          return make_pipeline(tf(), MLPClassifier(h, alpha=a, early_stopping=True, max_iter=200, random_state=SEED))
def cnb(a, norm, usa_tf):
    return make_pipeline(tf(), ComplementNB(alpha=a, norm=norm)) if usa_tf else ComplementNB(alpha=a, norm=norm)
def et(n, mf, msl, cw): return ExtraTreesClassifier(n, max_features=mf, min_samples_leaf=msl, class_weight=cw, n_jobs=N_THREADS, random_state=SEED)
def lgbm(n, eta, folhas, col=1.0, sub=1.0, mcs=20):
    return LGBMClassifier(n_estimators=n, learning_rate=eta, num_leaves=folhas, colsample_bytree=col, subsample=sub,
                          subsample_freq=1 if sub < 1 else 0, min_child_samples=mcs, n_jobs=N_THREADS, verbose=-1, random_state=SEED)
def lsvc(C, cw):        return make_pipeline(tf(), LinearSVC(C=C, class_weight=cw))

ESPACO = {   # posicao 0 = configuracao do experimento 02
  "Reg. Logística": [(lr, dict(C=C, cw=cw)) for C, cw in
                     [(10, "balanced"), (1, "balanced"), (3, "balanced"), (30, "balanced"), (100, "balanced"), (10, None), (1, None), (100, None)]],
  "SVM RBF":        [(svm, dict(k=k, C=C)) for k, C in [(300, 10), (300, 3), (300, 30), (150, 10), (150, 30), (500, 10), (300, 1), (500, 30)]],
  "kNN cosseno":    [(knn, dict(k=k, w=w)) for k, w in
                     [(5, "uniform"), (3, "uniform"), (7, "uniform"), (11, "uniform"), (15, "uniform"), (5, "distance"), (11, "distance"), (21, "distance")]],
  "MLP":            [(mlp, dict(h=h, a=a)) for h, a in [((128,), 1e-4), ((64,), 1e-4), ((256,), 1e-4), ((128,), 1e-3), ((128,), 1e-2), ((256, 128), 1e-4)]],
  "Naive Bayes":    [(cnb, dict(a=a, norm=n, usa_tf=t)) for a, n, t in
                     [(1, False, False), (0.1, False, False), (0.3, False, False), (3, False, False), (1, True, False), (0.1, True, False), (0.1, False, True), (1, False, True)]],
  "Extra Trees":    [(et, dict(n=n, mf=mf, msl=msl, cw=cw)) for n, mf, msl, cw in
                     [(400, "sqrt", 1, "balanced"), (400, "sqrt", 1, None), (400, "log2", 1, "balanced"), (400, 0.05, 1, "balanced"), (400, "sqrt", 2, "balanced"), (800, "sqrt", 1, "balanced")]],
  "LightGBM":       [(lgbm, kw) for kw in
                     [dict(n=400, eta=0.05, folhas=31), dict(n=200, eta=0.1, folhas=31), dict(n=800, eta=0.03, folhas=31), dict(n=400, eta=0.05, folhas=15),
                      dict(n=400, eta=0.05, folhas=63), dict(n=400, eta=0.05, folhas=31, col=0.3), dict(n=800, eta=0.05, folhas=15, col=0.3, sub=0.8),
                      dict(n=400, eta=0.05, folhas=31, mcs=5)]],
}
LINEARSVC = [(lsvc, dict(C=C, cw=cw)) for C, cw in [(1, "balanced"), (0.1, "balanced"), (0.3, "balanced"), (3, "balanced"), (10, "balanced"), (1, None), (0.3, None), (3, None)]]
META_C = [1.0, 0.03, 0.1, 0.3, 3.0, 10.0]
if SMOKE:
    ESPACO = {f: v[:2] for f, v in ESPACO.items()}; LINEARSVC = LINEARSVC[:2]; META_C = META_C[:2]
FAMILIAS = list(ESPACO)
TODOS = {**ESPACO, "LinearSVC": LINEARSVC}
FID = {f: i for i, f in enumerate(TODOS)}
CARD = [len(ESPACO[f]) for f in FAMILIAS] + [len(META_C)]
desc = lambda fam, k: ", ".join(f"{a}={b}" for a, b in TODOS[fam][k][1].items())

print(pd.DataFrame([dict(familia=f, opcoes=len(v), padrao=desc(f, 0)) for f, v in TODOS.items()]
                   + [dict(familia="meta-modelo (C)", opcoes=len(META_C), padrao=f"C={META_C[0]}")]).to_string(index=False))
print(f"\ncombinações possíveis do Stacking 7: {int(np.prod(CARD)):,}".replace(",", "."))

        familia  opcoes                             padrao
 Reg. Logística       8                  C=10, cw=balanced
        SVM RBF       8                        k=300, C=10
    kNN cosseno       8                     k=5, w=uniform
            MLP       6                 h=(128,), a=0.0001
    Naive Bayes       8      a=1, norm=False, usa_tf=False
    Extra Trees       6 n=400, mf=sqrt, msl=1, cw=balanced
       LightGBM       8         n=400, eta=0.05, folhas=31
      LinearSVC       8                   C=1, cw=balanced
meta-modelo (C)       6                              C=1.0

combinações possíveis do Stacking 7: 7.077.888


## 3. Motor: treina cada configuração uma vez por divisão e guarda as notas em cache

In [4]:
def nota(m, A):
    return m.predict_proba(A)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(A)

def arq(chave, fam, k):
    return CACHE / f"{chave}__f{FID[fam]}__k{k}.npz"

def tarefa(chave, Xtr, ytr, gtr, tes, si, fam, k):
    """Nota out-of-fold no treino (CV interna por grupo, igual para todas as configuracoes) + nota em cada teste."""
    destino = arq(chave, fam, k)
    if destino.exists(): return
    f, kw = TODOS[fam][k]; t0 = time.time(); ztr = np.zeros(Xtr.shape[0])
    for itr, ite in StratifiedGroupKFold(5, shuffle=True, random_state=si).split(Xtr, ytr, gtr):
        m = f(**kw).fit(Xtr[itr], ytr[itr]); ztr[ite] = nota(m, Xtr[ite])
    m = f(**kw).fit(Xtr, ytr)
    tmp = destino.with_name(destino.stem + "_tmp.npz")
    np.savez(tmp, ztr=ztr, t=time.time() - t0, **{f"zte_{n}": nota(m, A) for n, A in tes.items()})
    os.replace(tmp, destino)

SPLITS = {}   # chave -> dict(base, tr, tes, si)
def args(chave, fam, k):
    sp = SPLITS[chave]; Xb, yb, gb = BASES[sp["base"]]; tr = sp["tr"]
    tes = {n: (Xb[v] if isinstance(v, np.ndarray) and v.ndim == 1 else v) for n, v in sp["tes"].items()}
    return chave, Xb[tr], yb[tr], gb[tr], tes, sp["si"], fam, k

LENTO = {"Extra Trees": 0, "LightGBM": 1, "MLP": 2, "SVM RBF": 3}
def rodar(tarefas, rotulo, blocos=8):
    """Executa em blocos; a cada bloco escolhe quantos processos usar pela RAM livre."""
    global N_THREADS
    pend = sorted([t for t in tarefas if not arq(*t).exists()], key=lambda t: LENTO.get(t[1], 9))
    log(f"{rotulo}: {len(tarefas)} tarefas, {len(pend)} pendentes")
    t_ini = time.time()
    for q in range(blocos):
        bloco = pend[q::blocos]
        if not bloco: continue
        livre = psutil.virtual_memory().available / 2**30
        nw = 4 if livre > 5 else 3 if livre > 3.5 else 2 if livre > 2.5 else 1
        N_THREADS = 4 if nw == 1 else 2; t0 = time.time()
        if nw == 1:
            for t in bloco: tarefa(*args(*t))
        else:
            with parallel_config(backend="loky", inner_max_num_threads=N_THREADS):
                Parallel(n_jobs=nw)(delayed(tarefa)(*args(*t)) for t in bloco)
        gc.collect()
        log(f"{rotulo}: bloco {q + 1}/{blocos} ok | {len(bloco)} tarefas | {nw} em paralelo | RAM livre {livre:.1f} GB | "
            f"{time.time() - t0:.0f}s (total {time.time() - t_ini:.0f}s)")

def carregar(chave, fam, k, te):
    z = np.load(arq(chave, fam, k)); return z["ztr"], z[f"zte_{te}"]

def mcc_np(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    tp = np.sum(a & b); tn = np.sum(~a & ~b); fp = np.sum(~a & b); fn = np.sum(a & ~b)
    den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return (tp * tn - fp * fn) / den if den else 0.0

In [5]:
# Os dois Stacking 7 (exp. 03/04) e o LinearSVC afinado, por indice no espaco de busca, conferidos por descricao.
PADRAO = tuple([0] * len(CARD))
melhor_ga = (1, 7, 1, 1, 5, 5, 4, 5)       # Especialista = GA do exp. 03 (#5 na corrida)
lsvc_best = 5                              # LinearSVC C=1 sem peso de classe
esperado = ["C=1, cw=balanced", "k=500, C=30", "k=3, w=uniform", "h=(64,), a=0.0001", "a=0.1, norm=True, usa_tf=False",
            "n=800, mf=sqrt, msl=1, cw=balanced", "n=400, eta=0.05, folhas=63"]
assert [desc(f, melhor_ga[i]) for i, f in enumerate(FAMILIAS)] == esperado and META_C[melhor_ga[-1]] == 10.0
assert desc("LinearSVC", lsvc_best) == "C=1, cw=None"
def meta(C): return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
def t_corrigido(a, b, n_tr, n_te):
    d = np.asarray(a) - np.asarray(b); J = len(d); v = d.var(ddof=1)
    t = d.mean() / np.sqrt((1 / J + n_te / n_tr) * v) if v > 0 else np.inf
    return d.mean(), t, 2 * stats.t.sf(abs(t), J - 1)

In [6]:
METODOS = {"Stacking 7 padrão": ("stack", PADRAO), "Stacking 7 otimizado (GA)": ("stack", melhor_ga),
           "LinearSVC padrão": ("lsvc", 0), "LinearSVC otimizado": ("lsvc", lsvc_best)}
PRECISA = sorted({(fam, 0) for fam in FAMILIAS} | {(fam, melhor_ga[i]) for i, fam in enumerate(FAMILIAS)}
                 | {("LinearSVC", 0), ("LinearSVC", lsvc_best)}, key=str)

def limiar_custo(s, yv):
    """Limiar que minimiza 9 x (reais na lixeira) + 1 x (spams que passaram), escolhido so com o treino."""
    o = np.argsort(-s); ss, yy = s[o], yv[o]
    custo = CUSTO_FP * np.cumsum(yy == 0) + CUSTO_FN * ((yy == 1).sum() - np.cumsum(yy == 1))
    i = int(np.argmin(custo))
    if CUSTO_FN * (yy == 1).sum() <= custo[i]: return np.inf
    return (ss[i] + ss[i + 1]) / 2 if i + 1 < len(ss) else ss[i] - 1e-9

def prever(metodo, chave, te):
    """(nota, decisao no limiar padrao, decisao no limiar de custo)."""
    tipo, cfg = METODOS[metodo]; sp = SPLITS[chave]; ytr = BASES[sp["base"]][1][sp["tr"]]
    if tipo == "lsvc":
        ztr, zte = carregar(chave, "LinearSVC", cfg, te)
        return zte, (zte > 0).astype(int), (zte > limiar_custo(ztr, ytr)).astype(int)
    par = [carregar(chave, fam, cfg[i], te) for i, fam in enumerate(FAMILIAS)]
    Ztr, Zte = np.column_stack([a for a, _ in par]), np.column_stack([b for _, b in par])
    m = meta(META_C[cfg[-1]])
    oof = cross_val_predict(m, Ztr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    p = m.fit(Ztr, ytr).predict_proba(Zte)[:, 1]
    return p, (p > 0.5).astype(int), (p > limiar_custo(oof, ytr)).astype(int)

def metricas(yv, s, yh, yc):
    fp, fn = int(((yc == 1) & (yv == 0)).sum()), int(((yc == 0) & (yv == 1)).sum())
    return dict(mcc=matthews_corrcoef(yv, yh), pr_auc=average_precision_score(yv, s),
                real_lixeira=(yh[yv == 0] == 1).mean(), spam_passou=(yh[yv == 1] == 0).mean(),
                mcc_custo=matthews_corrcoef(yv, yc), real_lixeira_custo=(yc[yv == 0] == 1).mean(),
                spam_passou_custo=(yc[yv == 1] == 0).mean(), custo_por_mil=1000 * (CUSTO_FP * fp + CUSTO_FN * fn) / len(yv))

CONF_SEEDS = [201] if SMOKE else [201, 202, 203, 204]
for s in CONF_SEEDS:
    for f, (tr, te) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=s).split(X, y, g)):
        if SMOKE and f >= 2: break
        SPLITS[f"c{s}_{f}"] = dict(base="dev", tr=tr, tes={"te": te}, si=s * 100 + f)
CONF = [c for c in SPLITS if c.startswith("c")]
rodar([(c, fam, k) for c in CONF for fam, k in PRECISA], "confirmação", blocos=4)

linhas = []
for c in CONF:
    yte = y[SPLITS[c]["tes"]["te"]]
    for mt in METODOS:
        linhas.append(dict(metodo=mt, split=c, n_tr=len(SPLITS[c]["tr"]), n_te=len(yte), **metricas(yte, *prever(mt, c, "te"))))
R = pd.DataFrame(linhas)
pct = ["real_lixeira", "spam_passou", "real_lixeira_custo", "spam_passou_custo"]
res = R.groupby("metodo")[["mcc", "pr_auc", "real_lixeira", "spam_passou", "mcc_custo", "real_lixeira_custo", "spam_passou_custo", "custo_por_mil"]].mean()
res[pct] *= 100
res.loc[list(METODOS)].round(4)

confirmação: 320 tarefas, 0 pendentes


,mcc,pr_auc,real_lixeira,spam_passou,mcc_custo,real_lixeira_custo,spam_passou_custo,custo_por_mil
metodo,,,,,,,,
Stacking 7 padrão,0.9734,0.9973,0.9107,1.5625,0.9554,0.4085,5.3125,41.5130
Stacking 7 otimizado (GA),0.9744,0.9976,0.7746,1.7500,0.9615,0.3660,4.5625,36.6179
LinearSVC padrão,0.9600,0.9952,1.7364,1.5000,0.9392,0.4767,7.4375,52.0293
LinearSVC otimizado,0.9627,0.9956,1.1916,2.3958,0.9406,0.4511,7.3125,50.0340


## 7. Notas do TabPFN e conferência das divisões

In [7]:
S = np.load("splits_tabpfn.npz")
for c in CONF:   # o TabPFN usou exatamente os mesmos e-mails que o motor
    assert np.array_equal(dev[SPLITS[c]["tr"]], S[f"{c}__tr"]) and np.array_equal(dev[SPLITS[c]["tes"]["te"]], S[f"{c}__te"]), c
TAB = {}
for modo in ["nuvem", "local"]:
    for arq_ in sorted(Path(f"cache_tabpfn/{modo}").glob("*.npz")):
        if not arq_.stem.endswith("_tmp"):
            z = np.load(arq_); TAB[modo, arq_.stem] = {k: z[k] for k in z.files}
CONF_NUVEM = [c for c in CONF if ("nuvem", c) in TAB]
CONF_LOCAL = [c for c in CONF if ("local", c) in TAB]
log(f"notas do TabPFN | nuvem: {len(CONF_NUVEM)} divisões + externo={('nuvem', 'externo') in TAB} | local: {len(CONF_LOCAL)} divisões")

notas do TabPFN | nuvem: 20 divisões + externo=True | local: 5 divisões


## 8. Os seis métodos nas 20 divisões

In [8]:
def prever_tab(modo, chave, te):
    """TabPFN sozinho: (nota, decisao 0,5, decisao no limiar de custo escolhido com a nota OOF do treino)."""
    sp = SPLITS[chave]; ytr = BASES[sp["base"]][1][sp["tr"]]; z = TAB[modo, chave]
    p = z[f"zte_{te}"]
    return p, (p > 0.5).astype(int), (p > limiar_custo(z["ztr"], ytr)).astype(int)

def prever_stack8(gen, modo, chave, te):
    """Stacking 8 = os 7 do cromossomo + TabPFN como 8a coluna do meta-modelo."""
    sp = SPLITS[chave]; ytr = BASES[sp["base"]][1][sp["tr"]]
    par = [carregar(chave, fam, gen[i], te) for i, fam in enumerate(FAMILIAS)]
    z = TAB[modo, chave]
    Ztr = np.column_stack([a for a, _ in par] + [z["ztr"]]); Zte = np.column_stack([b for _, b in par] + [z[f"zte_{te}"]])
    m = meta(META_C[gen[-1]])
    oof = cross_val_predict(m, Ztr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    m.fit(Ztr, ytr); p = m.predict_proba(Zte)[:, 1]
    return p, (p > 0.5).astype(int), (p > limiar_custo(oof, ytr)).astype(int), m[-1].coef_.ravel()

M6 = ["TabPFN-3.5-Plus", "Stacking 7 padrão", "Stacking 7 especialista", "Stacking 8 padrão (+TabPFN)",
      "Stacking 8 especialista (+TabPFN)", "LinearSVC otimizado"]
METODOS["Stacking 7 especialista"] = ("stack", melhor_ga)
def prever6(nome, chave, te, modo="nuvem"):
    if nome.startswith("TabPFN"): return prever_tab(modo, chave, te), None
    if nome.startswith("Stacking 8"):
        *r, w = prever_stack8(PADRAO if "padrão" in nome else melhor_ga, modo, chave, te); return tuple(r), w
    return prever("Stacking 7 otimizado (GA)" if nome == "Stacking 7 especialista" else nome, chave, te), None

linhas, pesos8 = [], []
for c in CONF_NUVEM:
    yte = y[SPLITS[c]["tes"]["te"]]
    for nome in M6:
        r, w = prever6(nome, c, "te")
        linhas.append(dict(metodo=nome, split=c, n_tr=len(SPLITS[c]["tr"]), n_te=len(yte), **metricas(yte, *r)))
        if w is not None: pesos8.append(dict(metodo=nome, split=c, **dict(zip(FAMILIAS + ["TabPFN"], w))))
R5 = pd.DataFrame(linhas); R5.to_csv("tabpfn_20folds.csv", sep=";", decimal=",", index=False)
res5 = R5.groupby("metodo")[["mcc", "pr_auc", "real_lixeira", "spam_passou", "mcc_custo", "real_lixeira_custo", "spam_passou_custo", "custo_por_mil"]].mean()
res5[["real_lixeira", "spam_passou", "real_lixeira_custo", "spam_passou_custo"]] *= 100
print(f"{len(CONF_NUVEM)} divisões (sementes 201–204)")
res5.loc[M6].round(4)

20 divisões (sementes 201–204)


,mcc,pr_auc,real_lixeira,spam_passou,mcc_custo,real_lixeira_custo,spam_passou_custo,custo_por_mil
metodo,,,,,,,,
TabPFN-3.5-Plus,0.9625,0.9965,1.5661,1.5625,0.9224,0.4170,9.9792,55.5944
Stacking 7 padrão,0.9734,0.9973,0.9107,1.5625,0.9554,0.4085,5.3125,41.5130
Stacking 7 especialista,0.9744,0.9976,0.7746,1.7500,0.9615,0.3660,4.5625,36.6179
Stacking 8 padrão (+TabPFN),0.9727,0.9978,0.9022,1.6875,0.9565,0.3575,5.2917,38.1916
Stacking 8 especialista (+TabPFN),0.9747,0.9979,0.7745,1.7083,0.9572,0.3575,5.1875,37.8905
LinearSVC otimizado,0.9627,0.9956,1.1916,2.3958,0.9406,0.4511,7.3125,50.0340


In [9]:
rho = R5.n_te.mean() / R5.n_tr.mean()
def p_melhor(d):
    d = np.asarray(d); n = len(d); s = np.sqrt((1 / n + rho) * d.var(ddof=1))
    return float(1 - stats.t.cdf(0, n - 1, loc=d.mean(), scale=s)) if s > 0 else float(d.mean() > 0)
PARES5 = [("Stacking 8 padrão (+TabPFN)", "Stacking 7 padrão"), ("Stacking 8 especialista (+TabPFN)", "Stacking 7 especialista"),
          ("TabPFN-3.5-Plus", "LinearSVC otimizado"), ("Stacking 7 padrão", "TabPFN-3.5-Plus"),
          ("Stacking 8 especialista (+TabPFN)", "Stacking 7 padrão")]
lin = []
for met, sinal in [("mcc", 1), ("mcc_custo", 1), ("custo_por_mil", -1)]:
    P = R5.pivot(index="split", columns="metodo", values=met)
    for a_, b_ in PARES5:
        _, _, p = t_corrigido(P[a_], P[b_], R5.n_tr.mean(), R5.n_te.mean())
        eq = np.nan
        if met != "custo_por_mil":
            try: eq = baycomp.two_on_single(P[a_].values, P[b_].values, rope=0.01, runs=len(CONF_NUVEM) // 5)[1]
            except Exception: pass
        lin.append(dict(metrica=met, A=a_, B=b_, A_media=P[a_].mean(), B_media=P[b_].mean(), p=p,
                        **{"P(A melhor)": p_melhor(sinal * (P[a_] - P[b_])), "P(equiv.)": eq}))
print("P(A melhor) = probabilidade de A ser melhor por qualquer margem; P(equiv.) = |dif MCC| < 0,01")
pd.DataFrame(lin).round(4)

P(A melhor) = probabilidade de A ser melhor por qualquer margem; P(equiv.) = |dif MCC| < 0,01


,metrica,A,B,A_media,B_media,p,P(A melhor),P(equiv.)
0,mcc,Stacking 8 padrão (+TabPFN),Stacking 7 padrão,0.9727,0.9734,0.7225,0.3613,0.9998
1,mcc,Stacking 8 especialista (+TabPFN),Stacking 7 especialista,0.9747,0.9744,0.9081,0.5460,0.9990
2,mcc,TabPFN-3.5-Plus,LinearSVC otimizado,0.9625,0.9627,0.9740,0.4870,0.8680
3,mcc,Stacking 7 padrão,TabPFN-3.5-Plus,0.9734,0.9625,0.0681,0.9660,0.4342
4,mcc,Stacking 8 especialista (+TabPFN),Stacking 7 padrão,0.9747,0.9734,0.7096,0.6452,0.9898
5,mcc_custo,Stacking 8 padrão (+TabPFN),Stacking 7 padrão,0.9565,0.9554,0.8719,0.5640,0.8685
6,mcc_custo,Stacking 8 especialista (+TabPFN),Stacking 7 especialista,0.9572,0.9615,0.6157,0.3078,0.6928
7,mcc_custo,TabPFN-3.5-Plus,LinearSVC otimizado,0.9224,0.9406,0.1126,0.0563,0.2229
8,mcc_custo,Stacking 7 padrão,TabPFN-3.5-Plus,0.9554,0.9224,0.0027,0.9986,0.0133
9,mcc_custo,Stacking 8 especialista (+TabPFN),Stacking 7 padrão,0.9572,0.9554,0.8271,0.5865,0.7635


## 9. O TabPFN erra em e-mails diferentes? Pesos no meta-modelo e erros em comum

In [10]:
W8 = pd.DataFrame(pesos8)
print("peso médio de cada família no meta-modelo do Stacking 8 (unidades padronizadas):")
print(W8.groupby("metodo")[FAMILIAS + ["TabPFN"]].mean().T.round(2).to_string())

# erros em comum nas notas out-of-fold do treino (todas as 20 divisoes juntas)
err = {n: [] for n in FAMILIAS + ["TabPFN"]}; ys = []
for c in CONF_NUVEM:
    ytr = y[SPLITS[c]["tr"]]; ys.append(ytr)
    for i, fam in enumerate(FAMILIAS):
        z = carregar(c, fam, 0, "te")[0]; lim = 0.0 if fam == "SVM RBF" else 0.5   # SVC sem proba: nota = distancia a margem
        err[fam].append((z > lim) != ytr)
    err["TabPFN"].append((TAB["nuvem", c]["ztr"] > 0.5) != ytr)
E_ = pd.DataFrame({n: np.concatenate(v) for n, v in err.items()})
so_tab = (E_["TabPFN"] & ~E_[FAMILIAS].any(axis=1)).sum(); tab_salva = (~E_["TabPFN"] & E_[FAMILIAS].sum(axis=1).ge(4)).sum()
print(f"\nerros do TabPFN: {E_['TabPFN'].sum()} | só o TabPFN erra: {so_tab} | TabPFN acerta onde a maioria (≥4 de 7) erra: {tab_salva}")
print("taxa de erro em comum (Jaccard) do TabPFN com cada família (configuração padrão):")
print(pd.Series({f: (E_[f] & E_["TabPFN"]).sum() / (E_[f] | E_["TabPFN"]).sum() for f in FAMILIAS}).round(3).sort_values().to_string())

peso médio de cada família no meta-modelo do Stacking 8 (unidades padronizadas):
metodo          Stacking 8 especialista (+TabPFN)  Stacking 8 padrão (+TabPFN)
Reg. Logística                              -0.86                         0.72
SVM RBF                                      2.35                         0.81
kNN cosseno                                  1.40                         1.54
MLP                                          0.48                         0.42
Naive Bayes                                 -0.02                        -0.09
Extra Trees                                  0.47                         0.60
LightGBM                                     0.65                         0.57
TabPFN                                       1.97                         1.62

erros do TabPFN: 1107 | só o TabPFN erra: 37 | TabPFN acerta onde a maioria (≥4 de 7) erra: 428
taxa de erro em comum (Jaccard) do TabPFN com cada família (configuração padrão):
Naive Bayes       0.174
kNN c

## 10. Base externa: Enron2–6 (treino nos 5172 e-mails)

In [11]:
if ("nuvem", "externo") in TAB:
    EXT = next((p for p in [Path("../dados_externos/enron_vetores.npz")] if p.exists()), None)
    z = np.load(EXT); yext = {f"Enron{k}": z[f"y{k}"][S[f"ext_Enron{k}"]] for k in range(2, 7)}
    SPLITS["externo"] = dict(base="all", tr=np.arange(len(y_all)), tes={b: None for b in yext}, si=SEED)
    lin = []
    for b in yext:
        for nome in M6:
            r, _ = prever6(nome, "externo", b)
            lin.append(dict(base=b, metodo=nome, **metricas(yext[b], *r)))
    E5 = pd.DataFrame(lin); E5.to_csv("tabpfn_externo.csv", sep=";", decimal=",", index=False)
    for col, nome, pc in [("mcc_custo", "MCC, limiar de custo", False), ("pr_auc", "PR-AUC", False), ("custo_por_mil", "custo por mil e-mails, limiar de custo", False),
                          ("real_lixeira_custo", "% de e-mails reais na lixeira, limiar de custo", True), ("spam_passou_custo", "% de spam que passou, limiar de custo", True)]:
        t = E5.pivot(index="metodo", columns="base", values=col).reindex(M6); t["média"] = t.mean(1)
        print(f"\n== {nome} ==\n" + (t * 100 if pc else t).round(1 if (pc or col == "custo_por_mil") else 3).to_string())
else:
    E5 = None; print("notas externas do TabPFN ainda não geradas")


== MCC, limiar de custo ==
base                               Enron2  Enron3  Enron4  Enron5  Enron6  média
metodo                                                                          
TabPFN-3.5-Plus                     0.793   0.908   0.834   0.672   0.809  0.803
Stacking 7 padrão                   0.844   0.917   0.926   0.848   0.920  0.891
Stacking 7 especialista             0.832   0.913   0.936   0.861   0.931  0.894
Stacking 8 padrão (+TabPFN)         0.841   0.934   0.912   0.813   0.908  0.882
Stacking 8 especialista (+TabPFN)   0.839   0.937   0.905   0.787   0.899  0.873
LinearSVC otimizado                 0.822   0.914   0.841   0.735   0.833  0.829

== PR-AUC ==
base                               Enron2  Enron3  Enron4  Enron5  Enron6  média
metodo                                                                          
TabPFN-3.5-Plus                     0.952   0.990   0.998   0.995   0.999  0.987
Stacking 7 padrão                   0.936   0.990   0.997   0.994  

## 11. Réplica local (TabPFN-3.5-Fast em CPU) × nuvem (3.5-Plus), nas mesmas divisões

In [12]:
if CONF_LOCAL:
    lin = []
    for c in CONF_LOCAL:
        yte = y[SPLITS[c]["tes"]["te"]]
        for modo, rot in [("nuvem", "TabPFN-3.5-Plus (nuvem)"), ("local", "TabPFN-3.5-Fast (local)")]:
            lin.append(dict(metodo=rot, split=c, **metricas(yte, *prever_tab(modo, c, "te"))))
            *r, _ = prever_stack8(PADRAO, modo, c, "te")
            lin.append(dict(metodo=f"Stacking 8 padrão + {rot}", split=c, **metricas(yte, *r)))
        lin.append(dict(metodo="Stacking 7 padrão", split=c, **metricas(yte, *prever("Stacking 7 padrão", c, "te"))))
    RL = pd.DataFrame(lin); RL.to_csv("tabpfn_local_x_nuvem.csv", sep=";", decimal=",", index=False)
    print(f"{len(CONF_LOCAL)} divisões (semente 201)")
    print(RL.groupby("metodo")[["mcc", "pr_auc", "mcc_custo", "custo_por_mil"]].mean().round(4).to_string())
    corr = np.mean([np.corrcoef(TAB["local", c]["zte_te"], TAB["nuvem", c]["zte_te"])[0, 1] for c in CONF_LOCAL])
    print(f"\ncorrelação média entre as notas local e nuvem nos e-mails de teste: {corr:.3f}")
else:
    print("réplica local ainda não executada (rodar experimentos/rodar_noite_tabpfn.ps1)")
log("fim")

5 divisões (semente 201)
                                                mcc  pr_auc  mcc_custo  custo_por_mil
metodo                                                                               
Stacking 7 padrão                            0.9720  0.9969     0.9424        46.8932
Stacking 8 padrão + TabPFN-3.5-Fast (local)  0.9725  0.9969     0.9430        42.7840
Stacking 8 padrão + TabPFN-3.5-Plus (nuvem)  0.9714  0.9975     0.9511        47.1359
TabPFN-3.5-Fast (local)                      0.9565  0.9961     0.9125        55.3587
TabPFN-3.5-Plus (nuvem)                      0.9613  0.9963     0.9146        64.0590

correlação média entre as notas local e nuvem nos e-mails de teste: 0.993
fim
